# spancy_shift_spatial_clean.py — equivalence check against spancy_shift_spatial.py

`spancy_shift_spatial_clean.py` is the self-contained, trimmed copy of `spancy_shift_spatial.py`
(Stage 1 + model copied in from `spancy_shift.py`, unused code removed). This notebook checks that it
gives the same results as the original:

1. **§2 Stage 1 + graph:** `normalized_base`, `X_scaled`, scaler, bimodal mask, batch codes, regions,
   k-NN matrix must be **bit-identical**.
2. **§3 initial weights:** same `torch.manual_seed` → identical state_dicts (same keys).
3. **§4 training on CPU (deterministic):** 1 epoch × 5 steps, same seed → losses and weights must be
   **bit-identical**.
4. **§5 inference on GPU:** the same weights in both modules → corrections must agree up to the GPU
   run-to-run noise floor (measured in the same cell by running the clean module twice).
5. **§6 CLI smoke test:** `python spancy_shift_spatial_clean.py ... --epochs 1`.

Upload `spancy_shift.py`, `spancy_shift_spatial.py` (original) and `spancy_shift_spatial_clean.py`.
~15 min on a GPU runtime. **Paste back §7 (and any cell that fails).**

**Status (2026-10-12): ALL PASS** — the clean copy then REPLACED `spancy_shift_spatial.py`. To re-run:
the original is `git show d0f5fde:spancy_shift_spatial.py`, the clean copy is the current
`spancy_shift_spatial.py` (upload it as `spancy_shift_spatial_clean.py`).

## 0. Colab Setup

In [ ]:
# Install torch-geometric (auto-detect Colab's torch + CUDA version)
import torch
tv = torch.__version__.split('+')[0]
cv = torch.version.cuda or 'cpu'
if cv != 'cpu':
    cv = 'cu' + cv.replace('.', '')
whl = f'https://data.pyg.org/whl/torch-{tv}+{cv}.html'
print(f'PyTorch {tv}, CUDA tag {cv}')
!pip install -q torch-scatter torch-sparse torch-cluster torch-geometric -f {whl}
!pip install -q "pandas<3" "anndata<0.11"
print('Done.')

In [ ]:
from google.colab import files
import os

NEEDED = ['spancy_shift.py', 'spancy_shift_spatial.py', 'spancy_shift_spatial_clean.py']
missing = [f for f in NEEDED if not os.path.exists(f)]
if missing:
    print(f'Please upload: {missing}')
    files.upload()
for f in NEEDED:
    assert os.path.exists(f), f'{f} not found — upload it (Colab renames re-uploads to "name (1).py")'
    print(f'{f:32s} {os.path.getsize(f):>10,} bytes')
assert 'from spancy_shift import' not in open('spancy_shift_spatial_clean.py').read(), 'clean module still imports spancy_shift'
print('Modules: OK')

In [ ]:
import gc
import numpy as np
import torch

import spancy_shift_spatial as old          # original (imports spancy_shift)
import spancy_shift_spatial_clean as new    # self-contained clean copy

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
RESULTS = {}

In [ ]:
!wget -q https://zenodo.org/records/16383766/files/PRAD_anndata.h5ad
print('Download complete.')

## 1. Load data

In [ ]:
adata = new.load_adata('PRAD_anndata.h5ad')
adata_o = old.load_adata('PRAD_anndata.h5ad')
RESULTS['load_adata identical'] = (list(adata.var_names) == list(adata_o.var_names)
                                   and adata.obs.equals(adata_o.obs))
del adata_o
gc.collect()
print(adata)
print('load_adata identical:', RESULTS['load_adata identical'])

## 2. Stage 1 + scaling + graph (must be bit-identical)

In [ ]:
prep_o = old.prepare(adata, k_neighbors=15, bimodal_prominence=0.05, bimodal_min_batch_frac=0.5)
prep_n = new.prepare(adata, k_neighbors=15, bimodal_prominence=0.05, bimodal_min_batch_frac=0.5)

checks = {
    'normalized_base': np.array_equal(np.asarray(prep_o.adata_base.layers['normalized_base']),
                                      np.asarray(prep_n.adata_base.layers['normalized_base'])),
    'X_scaled': np.array_equal(prep_o.X_scaled, prep_n.X_scaled),
    'scaler center/scale': (np.array_equal(prep_o.scaler.center_, prep_n.scaler.center_)
                            and np.array_equal(prep_o.scaler.scale_, prep_n.scaler.scale_)),
    'is_bimodal': np.array_equal(prep_o.is_bimodal, prep_n.is_bimodal),
    'batch_codes': np.array_equal(prep_o.batch_codes, prep_n.batch_codes),
    'regions': np.array_equal(prep_o.regions, prep_n.regions),
    'knn': np.array_equal(prep_o.adj.knn, prep_n.adj.knn),
}
for k, v in checks.items():
    print(f'  {k:22s} {v}')
RESULTS['Stage 1 + graph bit-identical'] = all(checks.values())
print('bimodal markers:', [m for m, b in zip(adata.var_names, prep_n.is_bimodal) if b])

## 3. Initial weights (same seed → identical)

In [ ]:
n_batches = int(prep_n.batch_codes.max()) + 1
torch.manual_seed(6)
m_o = old.GNNStage2(n_markers=adata.n_vars, n_batches=n_batches, hidden=128, latent=64, n_heads=4)
torch.manual_seed(6)
m_n = new.GNNStage2(n_markers=adata.n_vars, n_batches=n_batches, hidden=128, latent=64, n_heads=4)
sd_o, sd_n = m_o.state_dict(), m_n.state_dict()
same_keys = list(sd_o.keys()) == list(sd_n.keys())
same_vals = same_keys and all(torch.equal(sd_o[k], sd_n[k]) for k in sd_o)
print(f'state_dict keys identical: {same_keys}   values identical: {same_vals}   ({len(sd_o)} tensors)')
RESULTS['initial weights identical'] = same_keys and same_vals
del m_o, m_n, sd_o, sd_n

## 4. Training on CPU, 1 epoch × 5 steps (must be bit-identical)

The clean module has no `max_steps_per_epoch`; for this test only, its sampler length is patched to 5.

In [ ]:
CFG = dict(n_epochs=1, lr=1e-3, weight_decay=1e-4, warmup_epochs=1, n_per_batch=512, gnn_hidden=128,
           gnn_latent=64, gnn_heads=4, temperature=0.07, w_recon=0.1, w_contrast=0.5, w_adv=0.3,
           w_mmd=1.0, mmd_samples=256, grl_max=1.0)
N_STEPS = 5

torch.manual_seed(6)
m_o, h_o = old.train(prep_o, device_str='cpu', max_steps_per_epoch=N_STEPS, **CFG)

_orig_len = new.NeighbourPatchSampler.__len__
new.NeighbourPatchSampler.__len__ = lambda self: min(_orig_len(self), N_STEPS)
try:
    torch.manual_seed(6)
    m_n, h_n = new.train(prep_n, device_str='cpu', **CFG)
finally:
    new.NeighbourPatchSampler.__len__ = _orig_len

hist_keys = ['loss', 'recon', 'contrast', 'adv', 'mmd', 'lr', 'grl_lambda']
for k in hist_keys:
    print(f'  {k:10s} old {h_o[k][0]:.10f}  new {h_n[k][0]:.10f}  equal {h_o[k] == h_n[k]}')
p_o = dict(m_o.named_parameters())
w_diff = max(float((p_o[n] - p).abs().max()) for n, p in m_n.named_parameters())
print(f'max |weight difference| after {N_STEPS} CPU steps: {w_diff:.3e}')
RESULTS['CPU training bit-identical (losses + weights)'] = all(h_o[k] == h_n[k] for k in hist_keys) and w_diff == 0.0
del m_o, m_n
gc.collect()

## 5. Inference on GPU, same weights in both modules

The clean module is trained 1 full epoch on the GPU (non-trivial weights); the same weights are
loaded into the original module's model. GPU message passing is not bit-reproducible, so the clean
module is also run twice to measure the noise floor.

In [ ]:
torch.manual_seed(6)
m_n, _ = new.train(prep_n, device_str=DEVICE, n_epochs=1, warmup_epochs=1)
m_o = old.GNNStage2(n_markers=adata.n_vars, n_batches=n_batches, hidden=128, latent=64, n_heads=4)
m_o.load_state_dict(m_n.state_dict())

d_o = old.correct(prep_o, m_o, device_str=DEVICE)
d_n = new.correct(prep_n, m_n, device_str=DEVICE)
d_n2 = new.correct(prep_n, m_n, device_str=DEVICE)
diff = float(np.abs(d_o - d_n).max())
floor = float(np.abs(d_n - d_n2).max())
L_o = np.log1p(old.apply(prep_o, d_o, alpha=0.6).astype(np.float64))
L_n = np.log1p(new.apply(prep_n, d_n, alpha=0.6).astype(np.float64))
log_diff = float(np.abs(L_o - L_n).max())
bim_max = float(np.abs(d_n[:, prep_n.is_bimodal]).max()) if prep_n.is_bimodal.any() else 0.0
out_n = new.normalize_adata(prep_n, m_n, alpha=0.6, device_str=DEVICE)
# normalize_adata re-runs correct() on the GPU -> compare in log1p space against the noise floor
nz_diff = float(np.abs(np.log1p(np.asarray(out_n.layers['normalized'], dtype=np.float64))
                       - np.log1p(new.apply(prep_n, d_n, alpha=0.6).astype(np.float64))).max())
nz_ok = nz_diff <= max(1e-5, 10 * floor)

print(f'max |delta| old vs clean     : {diff:.3e}')
print(f'max |delta| clean vs clean   : {floor:.3e}   (GPU noise floor)')
print(f'max |log1p output| old vs new: {log_diff:.3e}   (alpha 0.6)')
print(f'mean |delta| (correction size): {float(np.abs(d_n).mean()):.4f}')
print(f'max |delta| on bimodal markers: {bim_max:.3e}   (must be 0)')
print(f'normalize_adata vs apply(correct), max |log1p diff|: {nz_diff:.3e}  ok: {nz_ok}')
RESULTS['GPU inference old vs clean within 1e-5'] = diff <= max(1e-5, 10 * floor)
RESULTS['bimodal delta exactly 0'] = bim_max == 0.0
RESULTS['normalize_adata consistent'] = bool(nz_ok)
del out_n, L_o, L_n
gc.collect()

## 6. CLI smoke test (1 epoch)

In [ ]:
!python spancy_shift_spatial_clean.py --input PRAD_anndata.h5ad --output cli_out.h5ad --epochs 1 --device {DEVICE} --seed 6

In [ ]:
import anndata as ad
cli = ad.read_h5ad('cli_out.h5ad')
X_cli = np.asarray(cli.layers['normalized'])
cli_ok = (cli.shape == adata.shape
          and np.array_equal(np.asarray(cli.layers['normalized_base']),
                             np.asarray(prep_n.adata_base.layers['normalized_base']))
          and np.isfinite(X_cli).all() and (X_cli >= 0).all())
print(cli)
print('CLI output shape, Stage 1 layer, finite and non-negative:', cli_ok)
RESULTS['CLI runs and writes valid output'] = bool(cli_ok)
del cli, X_cli

## 7. Verdict — paste this back

In [ ]:
for k, v in RESULTS.items():
    print(f'  {"PASS" if v else "FAIL"}  {k}')
print('\nALL PASS' if all(RESULTS.values()) else '\nSOME CHECKS FAILED')